### Подготовка датасета по показателю объем убоя КРС

In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.stattools import kpss
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA
from pmdarima import auto_arima
from statsmodels.graphics.tsaplots import plot_acf
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error

from pylab import rcParams
from IPython.display import display
import math
from prophet import Prophet
pd.set_option('display.max_columns', 130)


import warnings
from statsmodels.tools.sm_exceptions import InterpolationWarning
warnings.simplefilter("ignore", category=InterpolationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)



In [11]:
df = pd.read_excel("../../Data cleansing/output data/Просуммированные по категориям с доп регрессорами.xlsx")
df.head(5)

,Показатель,Регион,2015-01,2015-02,2015-03,2015-04,2015-05,2015-06,2015-07,2015-08,2015-09,2015-10,2015-11,2015-12,2016-01,2016-02,2016-03,2016-04,2016-05,2016-06,2016-07,2016-08,2016-09,2016-10,2016-11,2016-12,2017-01,2017-02,2017-03,2017-04,2017-05,2017-06,2017-07,2017-08,2017-09,2017-10,2017-11,2017-12,2018-01,2018-02,2018-03,2018-04,2018-05,2018-06,2018-07,2018-08,2018-09,2018-10,2018-11,2018-12,2019-01,2019-02,2019-03,2019-04,2019-05,2019-06,2019-07,2019-08,2019-09,2019-10,2019-11,2019-12,2020-01,2020-02,2020-03,2020-04,2020-05,2020-06,2020-07,2020-08,2020-09,2020-10,2020-11,2020-12,2021-01,2021-02,2021-03,2021-04,2021-05,2021-06,2021-07,2021-08,2021-09,2021-10,2021-11,2021-12,2022-01,2022-02,2022-03,2022-04,2022-05,2022-06,2022-07,2022-08,2022-09,2022-10,2022-11,2022-12,2023-01,2023-02,2023-03,2023-04,2023-05,2023-06,2023-07,2023-08,2023-09,2023-10,2023-11,2023-12,2024-01,2024-02,2024-03,2024-04,2024-05,2024-06,2024-07,2024-08,2024-09,2024-10,2024-11,2024-12,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06,2025-07
0,Верблюды,АКМОЛИНСКАЯ ОБЛАСТЬ,0.00,0.00,0.40,0.00,0.00,0.00,0.09,1.00,0.00,0.00,0.20,0.00,0.00,0.18,0.28,0.00,0.00,0.40,0.00,0.00,0.65,0.00,0.31,1.00,0.00,0.00,0.00,0.00,0.00,2.01,0.00,1.20,0.00,0.00,2.18,0.39,0.00,0.00,1.04,0.00,0.14,2.08,0.00,0.00,0.00,0.00,0.00,0.30,0.00,0.00,0.00,0.66,0.00,0.33,0.00,0.9,0.00,0.00,0.00,10.08,0.00,0.0,0.00,0.0,0.00,0.54,0.0,0.0,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.36,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.0,0.00,0.0,0.00,0.40,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.0,0.00,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.0,0.00,0.0,0.0,0.00,0.0,0.00
1,Верблюды,АКТЮБИНСКАЯ ОБЛАСТЬ,101.98,67.47,374.84,115.59,218.72,14.15,19.77,3.00,39.16,46.16,238.56,463.35,109.04,72.94,384.96,114.35,221.89,10.93,18.79,3.50,38.53,46.64,216.08,472.07,110.32,68.17,380.15,127.12,217.61,14.98,21.26,7.14,44.63,51.78,239.43,513.33,115.70,69.40,371.53,130.17,218.41,14.59,21.76,7.19,45.97,55.48,252.68,527.66,117.94,70.37,385.75,117.80,218.45,15.54,21.78,7.2,47.06,58.02,257.54,543.35,119.80,71.3,396.90,121.8,228.10,15.90,21.9,7.2,63.00,59.90,260.20,552.10,121.60,71.8,398.40,128.50,228.80,13.10,22.50,7.20,63.90,61.90,150.70,554.90,121.30,73.3,404.20,128.70,234.90,13.60,23.60,6.3,49.80,62.4,153.70,552.8,118.07,69.17,384.21,121.01,223.46,11.76,21.39,5.52,48.18,61.39,150.87,531.79,119.50,72.60,391.0,126.1,229.10,12.2,24.10,5.7,49.87,63.00,154.3,538.40,129.3,77.60,411.5,128.1,235.70,12.2,25.17
2,Верблюды,АЛМАТИНСКАЯ ОБЛАСТЬ,1.00,0.20,51.60,25.40,0.00,61.90,76.87,95.57,16.09,0.10,10.80,46.78,15.06,13.70,131.80,9.20,17.89,130.12,2.00,0.00,4.99,2.06,4.90,44.48,15.06,23.94,41.60,19.54,3.00,15.14,18.50,14.82,14.52,12.60,9.92,43.11,20.72,2.96,15.24,0.00,1.00,20.03,3.44,7.77,117.25,1.00,0.00,20.34,21.95,3.04,17.91,26.81,9.16,18.80,13.37,0.0,17.29,15.90,4.69,23.10,10.85,9.7,42.75,0.0,6.21,21.10,4.0,2.6,28.10,4.53,79.50,93.18,23.70,11.2,16.00,2.00,3.10,19.62,2.30,4.85,11.18,2.80,9.65,36.44,10.60,11.5,26.10,7.20,12.75,11.20,11.20,27.0,25.26,17.9,21.90,12.5,16.55,16.81,22.78,12.10,12.44,6.85,39.65,17.22,8.85,27.48,0.48,29.72,18.90,17.40,12.5,16.4,11.60,16.7,15.70,24.0,7.70,6.90,5.4,12.60,22.6,19.00,8.2,12.0,40.70,41.6,28.10
3,Верблюды,АТЫРАУСКАЯ ОБЛАСТЬ,213.89,167.70,306.60,164.97,342.57,192.00,43.60,113.03,262.37,193.97,308.17,1087.33,325.10,190.60,301.10,154.84,328.50,220.30,66.10,118.20,234.90,196.51,270.92,974.18,303.30,167.41,323.54,182.34,349.10,249.40,57.57,88.40,221.21,199.63,278.80,964.02,305.22,159.62,326.87,159.50,367.20,257.50,60.01,78.10,253.27,332.00,355.05,989.86,280.08,165.05,334.51,134.85,367.97,321.22,126.22,93.7,263.10,366.66,348.54,1041.38,293.40,154.8,339.44,143.6,405.80,308.35,88.5,130.5,575.30,458.70,357.62,998.10,292.76,188.3,487.02,129.28,421.44,351.20,95.77,97.54,624.80,631.65,499.30,1179.66,286.67,189.8,513.16,171.46,402.10,432.22,121.53,126.2,637.50,651.7,482.94,1412.2,307.85,200.59,466.60,182.63,414.56,405.19,184.80,154.96,717.92,719.01

In [12]:
df_krs = df[df['Показатель'].isin(['КРС', 'Температура', 'Осадки', 'Поголовье: КРС', 'Цена: Говядина'])]
df_krs.sample(10)

,Показатель,Регион,2015-01,2015-02,2015-03,2015-04,2015-05,2015-06,2015-07,2015-08,2015-09,2015-10,2015-11,2015-12,2016-01,2016-02,2016-03,2016-04,2016-05,2016-06,2016-07,2016-08,2016-09,2016-10,2016-11,2016-12,2017-01,2017-02,2017-03,2017-04,2017-05,2017-06,2017-07,2017-08,2017-09,2017-10,2017-11,2017-12,2018-01,2018-02,2018-03,2018-04,2018-05,2018-06,2018-07,2018-08,2018-09,2018-10,2018-11,2018-12,2019-01,2019-02,2019-03,2019-04,2019-05,2019-06,2019-07,2019-08,2019-09,2019-10,2019-11,2019-12,2020-01,2020-02,2020-03,2020-04,2020-05,2020-06,2020-07,2020-08,2020-09,2020-10,2020-11,2020-12,2021-01,2021-02,2021-03,2021-04,2021-05,2021-06,2021-07,2021-08,2021-09,2021-10,2021-11,2021-12,2022-01,2022-02,2022-03,2022-04,2022-05,2022-06,2022-07,2022-08,2022-09,2022-10,2022-11,2022-12,2023-01,2023-02,2023-03,2023-04,2023-05,2023-06,2023-07,2023-08,2023-09,2023-10,2023-11,2023-12,2024-01,2024-02,2024-03,2024-04,2024-05,2024-06,2024-07,2024-08,2024-09,2024-10,2024-11,2024-12,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06,2025-07
295,Осадки,АЛМАТИНСКАЯ ОБЛАСТЬ,8.900000,21.300000,4.860000e+01,6.150000e+01,3.600000e+01,6.320000e+01,4.400000e+00,2.160000e+01,1.080000e+01,4.550000e+01,59.900000,27.500000,41.000000,6.200000,4.290000e+01,8.830000e+01,1.675000e+02,6.490000e+01,7.880000e+01,3.000000e-01,1.890000e+01,6.000000e+01,58.600000,60.700000,15.900000,31.100000,4.170000e+01,9.950000e+01,4.570000e+01,6.220000e+01,8.900000e+00,1.330000e+01,1.630000e+01,2.210000e+01,4.030000e+01,15.700000,26.700000,2.560000e+01,7.730000e+01,5.960000e+01,7.570000e+01,2.200000e+01,6.960000e+01,2.190000e+01,1.320000e+01,2.730000e+01,2.910000e+01,1.410000e+01,1.830000e+01,3.230000e+01,1.760000e+01,9.090000e+01,4.750000e+01,4.470000e+01,6.900000e+00,2.830000e+01,4.420000e+01,3.490000e+01,2.530000e+01,3.420000e+01,1.160000e+01,4.900000e+01,2.340000e+01,9.970000e+01,8.280000e+01,2.700000e+01,2.570000e+01,4.760000e+01,4.800000e+00,4.000000e+00,9.100000e+00,1.030000e+01,6.900000e+00,3.380000e+01,6.620000e+01,4.290000e+01,4.890000e+01,1.180000e+01,1.580000e+01,1.580000e+01,1.600000e+00,4.070000e+01,3.770000e+01,7.300000e+00,1.570000e+01,2.300000e+01,9.480000e+01,3.190000e+01,8.710000e+01,29.000000,12.200000,11.100000,1.500000,30.000000,66.700,7.400000,20.800000,20.000000,34.300000,37.100000,26.500000,9.400000,42.800000,40.800000,37.200000,51.300000,39.700000,48.000000,35.600000,37.300000,75.700000,73.400000,90.700000,14.200000,69.700000,6.800000,32.700000,60.400000,36.200000,30.300000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
195,Поголовье: КРС,КОСТАНАЙСКАЯ ОБЛАСТЬ,406856.000000,423599.000000,4.288610e+05,4.302560e+05,4.704070e+05,4.704840e+05,4.552040e+05,4.451290e+05,4.293180e+05,4.247420e+05,420203.000000,422367.000000,425787.000000,429592.000000,4.338990e+05,4.374540e+05,4.817500e+05,4.790730e+05,4.518180e+05,4.416100e+05,4.264870e+05,4.225460e+05,418661.000000,422065.000000,416867.000000,435182.000000,4.404310e+05,4.465370e+05,4.882440e+05,4.851460e+05,4.709730e+05,4.605360e+05,4.455290e+05,4.417190e+05,4.366980e+05,440720.000000,444833.000000,4.499150e+05,4.563000e+05,4.622130e+05,5.085730e+05,5.054660e+05,4.802770e+05,4.696580e+05,4.523230e+05,4.474470e+05,4.422810e+05,4.551660e+05,4.607520e+05,4.650880e+05,4.718010e+05,4.773340e+05,5.243400e+05,5.234140e+05,4.929050e+05,4.785750e+05,4.627220e+05,4.584940e+05,4.546150e+05,4.623680e+05,4.623230e+05,4.702100e+05,4.731570e+05,4.758680e+05,4.847400e+05,5.313350e+05,5.319160e+05,5.030940e+05,4.904490e+05,4.718580e+05,4.663720e+05,4.632560e+05,4.633740e+05,4.668810e+05,4.709490e+05,4.760700e+05,4.802600e+05,5.289960e+05,5.324910e+05,5.086340e+05,4.974150e+05,4.803250e+05,4.787030e+05,4.784850e+05,4.374140e+05,4.424640e+05,4.481610e+05,4.553540e+05,4.633260e+05,511968.000000,514095.000000,493695.000000,482741.000000,467236.000000,464278.000,463798.000000,114859.000000,113649.000000,112187.000000,113615.000000,113783.000000,113138.000000,111470.000000,109990.000000,109383.000000,110814.000000,114884.000000,116823.000000,376469.00000

In [13]:
# Step 1: Pivot to wide format (each indicator becomes columns of periods)
df_wide = df_krs.pivot(index="Регион", columns="Показатель")

# Step 2: Flatten multi-level columns: ('2015-01', 'КРС') → 'КРС_2015-01'
df_wide.columns = [f"{col[1]}_{col[0]}" for col in df_wide.columns]
df_wide = df_wide.reset_index()

# Step 3: Melt: one row per region-period-indicator
df_melted = df_wide.melt(id_vars="Регион", var_name="indicator_period", value_name="value")

# Step 4: Extract 'Период' and 'Показатель' from the combined column
df_melted["Период"] = df_melted["indicator_period"].str.extract(r"_(\d{4}-\d{2})$")
df_melted["Показатель"] = df_melted["indicator_period"].str.extract(r"^(.+)_\d{4}-\d{2}")

# Step 5: Pivot again to get final modeling format: one row per region+period, one column per indicator
df_krs = df_melted.pivot_table(index=["Регион", "Период"], columns="Показатель", values="value").reset_index()
print(df_krs.groupby("Регион").size().reset_index(name="Количество строк"))
df_krs

                            Регион  Количество строк
0              АКМОЛИНСКАЯ ОБЛАСТЬ               127
1              АКТЮБИНСКАЯ ОБЛАСТЬ               127
2              АЛМАТИНСКАЯ ОБЛАСТЬ               127
3               АТЫРАУСКАЯ ОБЛАСТЬ               127
4   ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ               127
5                          ГАЛМАТЫ               127
6                          ГАСТАНА               127
7                         ГШЫМКЕНТ                86
8               ЖАМБЫЛСКАЯ ОБЛАСТЬ               127
9    ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ               127
10          КАРАГАНДИНСКАЯ ОБЛАСТЬ               127
11            КОСТАНАЙСКАЯ ОБЛАСТЬ               127
12          КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ               127
13           МАНГИСТАУСКАЯ ОБЛАСТЬ               127
14                    ОБЛАСТЬ АБАЙ                38
15                  ОБЛАСТЬ ЖЕТІСУ                38
16                  ОБЛАСТЬ ҰЛЫТАУ                38
17            ПАВЛОДАРСКАЯ ОБЛАСТЬ            

Показатель,Регион,Период,КРС,Осадки,Поголовье: КРС,Температура,Цена: Говядина
0,АКМОЛИНСКАЯ ОБЛАСТЬ,2015-01,4455.35,9.8,372560.0,-12.490323,100.0
1,АКМОЛИНСКАЯ ОБЛАСТЬ,2015-02,3654.20,9.8,399442.0,-10.192857,100.0
2,АКМОЛИНСКАЯ ОБЛАСТЬ,2015-03,4287.08,8.3,425605.0,-5.870968,100.0
3,АКМОЛИНСКАЯ ОБЛАСТЬ,2015-04,3923.21,8.8,440023.0,4.490000,99.8
4,АКМОЛИНСКАЯ ОБЛАСТЬ,2015-05,3849.70,42.8,444647.0,14.574194,99.8
...,...,...,...,...,...,...,...
2313,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-03,8225.02,NaN,NaN,NaN,NaN
2314,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-04,6628.30,NaN,NaN,NaN,NaN
2315,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-05,7697.58,NaN,NaN,NaN,NaN
2316,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-06,9167.36,NaN,NaN,NaN,NaN


In [14]:
df_krs = df_krs[df_krs["Регион"] != 'РЕСПУБЛИКА КАЗАХСТАН']

In [15]:
df_krs

Показатель,Регион,Период,КРС,Осадки,Поголовье: КРС,Температура,Цена: Говядина
0,АКМОЛИНСКАЯ ОБЛАСТЬ,2015-01,4455.35,9.8,372560.0,-12.490323,100.0
1,АКМОЛИНСКАЯ ОБЛАСТЬ,2015-02,3654.20,9.8,399442.0,-10.192857,100.0
2,АКМОЛИНСКАЯ ОБЛАСТЬ,2015-03,4287.08,8.3,425605.0,-5.870968,100.0
3,АКМОЛИНСКАЯ ОБЛАСТЬ,2015-04,3923.21,8.8,440023.0,4.490000,99.8
4,АКМОЛИНСКАЯ ОБЛАСТЬ,2015-05,3849.70,42.8,444647.0,14.574194,99.8
...,...,...,...,...,...,...,...
2313,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-03,8225.02,NaN,NaN,NaN,NaN
2314,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-04,6628.30,NaN,NaN,NaN,NaN
2315,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-05,7697.58,NaN,NaN,NaN,NaN
2316,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-06,9167.36,NaN,NaN,NaN,NaN


In [16]:
df_krs.to_excel("Датасет по КРС с регрессорами.xlsx", index=False)

In [17]:
df_krs = df_krs.drop(columns=['Осадки', 'Поголовье: КРС', 'Температура',
       'Цена: Говядина'])
df_krs.sample(10)

Показатель,Регион,Период,КРС
1375,КОСТАНАЙСКАЯ ОБЛАСТЬ,2016-08,3439.29
38,АКМОЛИНСКАЯ ОБЛАСТЬ,2018-03,4088.16
1100,ЖАМБЫЛСКАЯ ОБЛАСТЬ,2025-06,6448.97
2169,СЕВЕРО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,2020-05,2768.09
1691,МАНГИСТАУСКАЯ ОБЛАСТЬ,2021-10,214.60
2221,СЕВЕРО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,2024-09,5090.14
1829,ОБЛАСТЬ ҰЛЫТАУ,2023-10,551.93
16,АКМОЛИНСКАЯ ОБЛАСТЬ,2016-05,3924.54
1061,ЖАМБЫЛСКАЯ ОБЛАСТЬ,2022-03,4490.67
1564,КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ,2021-10,2248.74


In [18]:
df_krs.to_excel("Датасет по КРС.xlsx", index=False)

#### Приведение в широкий формат (для back-end)

In [19]:
# 1) приведение типов
df = df_krs.copy()
df["Период"] = pd.to_datetime(df["Период"], format="%Y-%m", errors="coerce")
df["КРС"] = pd.to_numeric(df["КРС"], errors="coerce")

# 2) wide-таблица: строки — месяцы, столбцы — регионы, значения — КРС
df_krs_wide = (df
        .pivot_table(index="Период", columns="Регион", values="КРС", aggfunc="sum")  # если дублей нет — можно aggfunc="first"
        .sort_index())
# 3) Убираем лишний уровень индекса у колонок
df_krs_wide.columns.name = None

# 4) Возвращаем "Период" в строковый формат YYYY-MM
df_krs_wide = df_krs_wide.reset_index()
df_krs_wide["Период"] = df_krs_wide["Период"].dt.strftime("%Y-%m")

df_krs_wide.to_excel("Датасет по КРС wide.xlsx", index=False)
df_krs_wide

,Период,АКМОЛИНСКАЯ ОБЛАСТЬ,АКТЮБИНСКАЯ ОБЛАСТЬ,АЛМАТИНСКАЯ ОБЛАСТЬ,АТЫРАУСКАЯ ОБЛАСТЬ,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,ГАЛМАТЫ,ГАСТАНА,ГШЫМКЕНТ,ЖАМБЫЛСКАЯ ОБЛАСТЬ,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,КАРАГАНДИНСКАЯ ОБЛАСТЬ,КОСТАНАЙСКАЯ ОБЛАСТЬ,КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ,МАНГИСТАУСКАЯ ОБЛАСТЬ,ОБЛАСТЬ АБАЙ,ОБЛАСТЬ ЖЕТІСУ,ОБЛАСТЬ ҰЛЫТАУ,ПАВЛОДАРСКАЯ ОБЛАСТЬ,СЕВЕРО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,ТУРКЕСТАНСКАЯ ОБЛАСТЬ
0,2015-01,4455.35,5786.80,6087.47,1957.38,4151.27,35.38,7.48,NaN,2731.22,2406.30,4276.61,3858.83,1446.34,30.15,NaN,NaN,NaN,2646.74,6302.91,NaN
1,2015-02,3654.20,5425.85,4454.61,1755.18,6473.64,14.65,27.08,NaN,3109.06,2980.56,2663.85,5661.27,1165.32,225.79,NaN,NaN,NaN,2934.07,3236.66,NaN
2,2015-03,4287.08,6578.37,16005.48,2085.52,6837.39,34.19,13.91,NaN,2538.81,3497.07,3105.64,4313.48,1233.92,581.50,NaN,NaN,NaN,3230.62,1531.42,NaN
3,2015-04,3923.21,5130.34,3934.04,1346.93,5875.35,40.53,19.11,NaN,3379.88,3061.07,2216.56,4841.38,1355.81,574.71,NaN,NaN,NaN,2731.05,2111.45,NaN
4,2015-05,3849.70,5668.00,7098.30,2073.99,5952.33,36.70,17.31,NaN,2921.49,3678.77,3436.05,7257.34,1189.73,270.72,NaN,NaN,NaN,2830.10,2218.93,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122,2025-03,4315.53,6574.41,8660.45,1740.22,2809.92,0.30,2.70,400.39,2715.00,4863.53,2404.24,3137.83,1109.61,103.05,3152.38,7699.85,401.05,2864.20,2064.27,8225.02
123,2025-04,3934.11,5766.31,2975.60,1429.08,2204.76,0.00,2.00,403.40,3245.04,3999.99,1792.75,3184.39,1429.88,279.48,2634.39,2296.43,179.60,2872.67,1625.97,6628.30
124,2025-05,3766.97,5446.24,4823.44,1686.36,2856.12,0.00,2.10,435.75,3464.30,4231.86,2202.71,4128.53,1463.98,40.30,2757.33,2733.24,443.45,2706.78,1849.39,7697.58
125,2025-06,4346.53,7109.86,14601.52,1765.10,6147.46,45.50,2.60,517.70,6448.97,8235.66,6945.26,4248.12,1578.15,226.00,6965.91,9527.94,854.70,4220.25,1416.17,9167.36
